# MELD Dataset Analyzer
Systematically explores the structure of `meld_multi_features.pkl` to decode what every field, number, and array represents — and how they map to `meld_vlm_manifest.csv` for visual feature replacement.

## Section 1 — Import Required Libraries

In [1]:
import pickle
import numpy as np
import pandas as pd
import pprint
from collections import Counter
from pathlib import Path

PKL_PATH  = "Dataset/CFN-ESA/meld_multi_features.pkl"
MANIFEST  = "meld_vlm_manifest.csv"

print("Libraries loaded successfully.")
print(f"PKL path  : {PKL_PATH}")
print(f"Manifest  : {MANIFEST}")

Libraries loaded successfully.
PKL path  : Dataset/CFN-ESA/meld_multi_features.pkl
Manifest  : meld_vlm_manifest.csv


## Section 2 — Load the MELD PKL File

In [2]:
# Load with default encoding (MELDDataset_BERT does NOT pass encoding="latin1")
raw = pickle.load(open(PKL_PATH, "rb"))

print(f"Type of loaded object : {type(raw)}")
if isinstance(raw, (list, tuple)):
    print(f"Number of top-level fields : {len(raw)}")
    for i, item in enumerate(raw):
        t = type(item).__name__
        sz = len(item) if hasattr(item, "__len__") else "scalar"
        print(f"  [{i:2d}] type={t:10s}  len={sz}")
elif isinstance(raw, dict):
    print(f"Top-level keys : {list(raw.keys())}")
else:
    print(raw)

Type of loaded object : <class 'list'>
Number of top-level fields : 14
  [ 0] type=dict        len=1432
  [ 1] type=dict        len=1432
  [ 2] type=dict        len=1432
  [ 3] type=dict        len=1432
  [ 4] type=dict        len=1432
  [ 5] type=dict        len=1432
  [ 6] type=dict        len=1432
  [ 7] type=dict        len=1432
  [ 8] type=dict        len=1432
  [ 9] type=dict        len=1432
  [10] type=dict        len=1432
  [11] type=set         len=1152
  [12] type=set         len=280
  [13] type=NoneType    len=scalar


## Section 3 — Inspect Top-Level Structure & Unpack Fields
The pkl is a tuple of 14 items (matching `MELDDataset_BERT.__init__`):
`videoIDs, videoSpeakers, videoLabels, videoSentiments, videoText0-3, videoAudio, videoVisual, videoSentence, trainVid, testVid, _`

In [3]:
(
    videoIDs,
    videoSpeakers,
    videoLabels,
    videoSentiments,
    videoText0,
    videoText1,
    videoText2,
    videoText3,
    videoAudio,
    videoVisual,
    videoSentence,
    trainVid,
    testVid,
    _extra,
) = raw

# Overview of all dialogues
all_vids = sorted(videoIDs.keys())
total_utts = sum(len(videoIDs[v]) for v in all_vids)

print(f"Total dialogues  : {len(all_vids)}")
print(f"Train dialogues  : {len(trainVid)}")
print(f"Test  dialogues  : {len(testVid)}")
print(f"Total utterances : {total_utts}")
print()
print(f"Type of dialogue key  : {type(all_vids[0])}")
print(f"Sample dialogue IDs (first 15): {all_vids[:15]}")
print(f"Sample dialogue IDs (last  5) : {all_vids[-5:]}")
print()
print(f"_extra field  type={type(_extra).__name__}  len={len(_extra) if hasattr(_extra,'__len__') else 'scalar'}")

Total dialogues  : 1432
Train dialogues  : 1152
Test  dialogues  : 280
Total utterances : 13708

Type of dialogue key  : <class 'int'>
Sample dialogue IDs (first 15): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Sample dialogue IDs (last  5) : [1428, 1429, 1430, 1431, 1432]

_extra field  type=NoneType  len=scalar


### 3a. `videoIDs` — utterance IDs per dialogue

In [4]:
sample_vid = all_vids[0]

print(f"Sample dialogue key : {repr(sample_vid)}")
print(f"  Utterance IDs     : {videoIDs[sample_vid]}")
print(f"  Type of utt ID    : {type(videoIDs[sample_vid][0])}")
print(f"  # utterances      : {len(videoIDs[sample_vid])}")
print()

# Dialogue length distribution
lengths = [len(videoIDs[v]) for v in all_vids]
print(f"Dialogue length stats:")
print(f"  min={min(lengths)}, max={max(lengths)}, "
      f"mean={np.mean(lengths):.1f}, median={np.median(lengths):.0f}")
print()

# Show train vs test
train_utts = sum(len(videoIDs[v]) for v in trainVid)
test_utts  = sum(len(videoIDs[v]) for v in testVid)
print(f"trainVid : {len(trainVid):4d} dialogues  → {train_utts:5d} utterances")
print(f"testVid  : {len(testVid):4d} dialogues  → {test_utts:5d} utterances")
print()
print(f"First 5 train dialogue IDs : {list(trainVid)[:5]}")
print(f"First 5 test  dialogue IDs : {list(testVid)[:5]}")

Sample dialogue key : 0
  Utterance IDs     : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
  Type of utt ID    : <class 'int'>
  # utterances      : 14

Dialogue length stats:
  min=1, max=33, mean=9.6, median=9

trainVid : 1152 dialogues  → 11098 utterances
testVid  :  280 dialogues  →  2610 utterances

First 5 train dialogue IDs : [0, 1, 2, 3, 4]
First 5 test  dialogue IDs : [1153, 1154, 1155, 1156, 1157]


### 3b. `videoSpeakers` — speaker identity per utterance
In MELD, speakers are named (e.g. "Ross", "Monica") encoded as one-hot multi-speaker vectors rather than simple M/F.
The model receives the raw `videoSpeakers` array — see `MELDDataset_BERT.__getitem__` for the tensor conversion.

In [5]:
sample_spk = videoSpeakers[sample_vid]
print(f"Speaker data type for '{sample_vid}': {type(sample_spk)}")
print(f"First entry type: {type(sample_spk[0]) if hasattr(sample_spk,'__len__') else type(sample_spk)}")
print(f"Raw speakers for sample dialogue:")
print(f"  {sample_spk}")
print()

# Count all unique speaker representations across full dataset
all_speaker_entries = [s for v in all_vids for s in videoSpeakers[v]]
print(f"Total speaker entries    : {len(all_speaker_entries)}")
# Handle both string speakers and array speakers
s0 = all_speaker_entries[0]
if isinstance(s0, str):
    speaker_counter = Counter(all_speaker_entries)
    print("Speaker type: string labels")
    print("\nAll unique speakers and their utterance counts:")
    for spk, cnt in speaker_counter.most_common():
        print(f"  {spk:20s} : {cnt:5d}  ({100*cnt/len(all_speaker_entries):.1f}%)")
elif hasattr(s0, '__len__'):
    print(f"Speaker type: vector (len={len(s0)})")
    print(f"  sample entry: {s0}")
else:
    print(f"Speaker type: {type(s0)}")
    print(f"  sample entry: {s0}")

Speaker data type for '0': <class 'list'>
First entry type: <class 'list'>
Raw speakers for sample dialogue:
  [[1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0]]

Total speaker entries    : 13708
Speaker type: vector (len=9)
  sample entry: [1, 0, 0, 0, 0, 0, 0, 0, 0]


### 3c. `videoLabels` & `videoSentiments` — emotion and sentiment per utterance
MELD 7-class emotion mapping: `{0:neutral, 1:surprise, 2:fear, 3:sadness, 4:joy, 5:disgust, 6:anger}`  
Sentiments stored directly in `videoSentiments` (not derived on-the-fly like IEMOCAP).

In [6]:
EMOTION_MAP   = {0:'neutral', 1:'surprise', 2:'fear', 3:'sadness', 4:'joy', 5:'disgust', 6:'anger'}
SENTIMENT_MAP = {0:'negative', 1:'neutral', 2:'positive'}

emo_seq = videoLabels[sample_vid]
sen_seq = videoSentiments[sample_vid]

print(f"Labels for sample dialogue '{sample_vid}':")
for i, (uid, emo, sen) in enumerate(zip(videoIDs[sample_vid], emo_seq, sen_seq)):
    print(f"  [{i:2d}] id={uid}  emo={emo}({EMOTION_MAP[emo]:8s})  sentiment={sen}({SENTIMENT_MAP[sen]})")

print()
# Full dataset distributions
all_emo = [l for v in all_vids for l in videoLabels[v]]
all_sen = [l for v in all_vids for l in videoSentiments[v]]
emo_counts = Counter(all_emo)
sen_counts = Counter(all_sen)

print(f"Total utterances : {len(all_emo)}")
print("\nEmotion class distribution (full dataset):")
for k in sorted(emo_counts):
    print(f"  {k} ({EMOTION_MAP[k]:8s}) : {emo_counts[k]:5d}  ({100*emo_counts[k]/len(all_emo):.1f}%)")

print("\nSentiment distribution (full dataset):")
for k in sorted(sen_counts):
    print(f"  {k} ({SENTIMENT_MAP[k]:8s}) : {sen_counts[k]:5d}  ({100*sen_counts[k]/len(all_sen):.1f}%)")

Labels for sample dialogue '0':
  [ 0] id=0  emo=0(neutral )  sentiment=0(negative)
  [ 1] id=1  emo=0(neutral )  sentiment=0(negative)
  [ 2] id=2  emo=0(neutral )  sentiment=0(negative)
  [ 3] id=3  emo=0(neutral )  sentiment=0(negative)
  [ 4] id=4  emo=1(surprise)  sentiment=1(neutral)
  [ 5] id=5  emo=0(neutral )  sentiment=0(negative)
  [ 6] id=6  emo=0(neutral )  sentiment=0(negative)
  [ 7] id=7  emo=0(neutral )  sentiment=0(negative)
  [ 8] id=8  emo=0(neutral )  sentiment=0(negative)
  [ 9] id=9  emo=0(neutral )  sentiment=0(negative)
  [10] id=10  emo=2(fear    )  sentiment=2(positive)
  [11] id=11  emo=0(neutral )  sentiment=0(negative)
  [12] id=12  emo=1(surprise)  sentiment=1(neutral)
  [13] id=13  emo=0(neutral )  sentiment=0(negative)

Total utterances : 13708

Emotion class distribution (full dataset):
  0 (neutral ) :  6436  (47.0%)
  1 (surprise) :  1636  (11.9%)
  2 (fear    ) :   358  (2.6%)
  3 (sadness ) :  1002  (7.3%)
  4 (joy     ) :  2308  (16.8%)
  5 (disgu

## Section 4 — Analyze Feature Dimensions and Data Types
Examine the shape, dtype, and value range of text, visual, and audio features.

In [7]:
def feature_summary(name, field, vids):
    """Print shape, dtype and value stats for a feature field."""
    arr = np.array(field[vids[0]])
    print(f"{name}['{vids[0]}']")
    print(f"    shape   : {arr.shape}   (seq_len × feature_dim)")
    print(f"    dtype   : {arr.dtype}")
    print(f"    min     : {arr.min():.5f}")
    print(f"    max     : {arr.max():.5f}")
    print(f"    mean    : {arr.mean():.6f}")
    print(f"    std     : {arr.std():.6f}")
    # Check consistency of feature dim across all dialogues
    dims = set(np.array(field[v]).shape[1] for v in vids)
    print(f"    feature dims across all dialogues : {dims}")
    print()

print("=" * 65)
print("TEXT FEATURES")
print("=" * 65)
for name, field in [("videoText0", videoText0), ("videoText1", videoText1),
                    ("videoText2", videoText2), ("videoText3", videoText3)]:
    feature_summary(name, field, all_vids)

print("=" * 65)
print("VISUAL FEATURES  ← target for VLM replacement")
print("=" * 65)
feature_summary("videoVisual", videoVisual, all_vids)

print("=" * 65)
print("AUDIO FEATURES")
print("=" * 65)
feature_summary("videoAudio", videoAudio, all_vids)

# Summary table
text_dim    = np.array(videoText0[sample_vid]).shape[1]
visual_dim  = np.array(videoVisual[sample_vid]).shape[1]
audio_dim   = np.array(videoAudio[sample_vid]).shape[1]

print("=" * 65)
print("DIMENSION SUMMARY")
print("=" * 65)
print(f"  Text (BERT, 4 variants)  : {text_dim}-dim")
print(f"  Visual (to be replaced)  : {visual_dim}-dim   → will become 768-dim (Longformer)")
print(f"  Audio (unchanged)        : {audio_dim}-dim")

TEXT FEATURES
videoText0['0']
    shape   : (14, 1024)   (seq_len × feature_dim)
    dtype   : float32
    min     : -11.20238
    max     : 11.64803
    mean    : -0.031170
    std     : 0.991781
    feature dims across all dialogues : {1024}

videoText1['0']
    shape   : (14, 1024)   (seq_len × feature_dim)
    dtype   : float32
    min     : -12.27611
    max     : 4.16376
    mean    : -0.030885
    std     : 0.996217
    feature dims across all dialogues : {1024}

videoText2['0']
    shape   : (14, 1024)   (seq_len × feature_dim)
    dtype   : float32
    min     : -27.55914
    max     : 2.67241
    mean    : -0.041755
    std     : 1.003325
    feature dims across all dialogues : {1024}

videoText3['0']
    shape   : (14, 1024)   (seq_len × feature_dim)
    dtype   : float32
    min     : -30.92405
    max     : 2.09319
    mean    : -0.033421
    std     : 0.986514
    feature dims across all dialogues : {1024}

VISUAL FEATURES  ← target for VLM replacement
videoVisual['0']
  

## Section 5 — Explore Sample Entries
Display a few complete entries to see how all fields align for individual utterances.

In [8]:
print(f"Full walkthrough of sample dialogue: '{sample_vid}'")
print("=" * 75)
n_utts = len(videoIDs[sample_vid])

for i in range(n_utts):
    uid  = videoIDs[sample_vid][i]
    emo  = videoLabels[sample_vid][i]
    sen  = videoSentiments[sample_vid][i]
    text = videoSentence[sample_vid][i]

    # Speaker entry (could be string or vector)
    spk_raw = videoSpeakers[sample_vid][i] if hasattr(videoSpeakers[sample_vid], '__getitem__') else videoSpeakers[sample_vid]

    vis_vec  = np.array(videoVisual[sample_vid])[i]
    aud_vec  = np.array(videoAudio[sample_vid])[i]
    txt_vec  = np.array(videoText0[sample_vid])[i]

    print(f"  [{i:2d}] utt_id={repr(uid)}")
    print(f"        speaker    : {spk_raw}")
    print(f"        emotion    : {emo} ({EMOTION_MAP[emo]})")
    print(f"        sentiment  : {sen} ({SENTIMENT_MAP[sen]})")
    print(f"        sentence   : \"{text}\"")
    print(f"        text0 vec  : shape={txt_vec.shape}  range=[{txt_vec.min():.3f}, {txt_vec.max():.3f}]")
    print(f"        visual vec : shape={vis_vec.shape}  range=[{vis_vec.min():.3f}, {vis_vec.max():.3f}]")
    print(f"        audio vec  : shape={aud_vec.shape}  range=[{aud_vec.min():.3f}, {aud_vec.max():.3f}]")
    print()

Full walkthrough of sample dialogue: '0'
  [ 0] utt_id=0
        speaker    : [1, 0, 0, 0, 0, 0, 0, 0, 0]
        emotion    : 0 (neutral)
        sentiment  : 0 (negative)
        sentence   : "also I was the point person on my companys transition from the KL-5 to GR-6 system."
        text0 vec  : shape=(1024,)  range=[-3.808, 5.560]
        visual vec : shape=(342,)  range=[0.000, 2.059]
        audio vec  : shape=(300,)  range=[-1.000, 1.000]

  [ 1] utt_id=1
        speaker    : [0, 1, 0, 0, 0, 0, 0, 0, 0]
        emotion    : 0 (neutral)
        sentiment  : 0 (negative)
        sentence   : "You mustve had your hands full."
        text0 vec  : shape=(1024,)  range=[-3.035, 4.479]
        visual vec : shape=(342,)  range=[0.000, 2.486]
        audio vec  : shape=(300,)  range=[-1.000, 1.000]

  [ 2] utt_id=2
        speaker    : [1, 0, 0, 0, 0, 0, 0, 0, 0]
        emotion    : 0 (neutral)
        sentiment  : 0 (negative)
        sentence   : "That I did. That I did."
        

## Section 6 — Load the MELD VLM Manifest CSV
Inspect `meld_vlm_manifest.csv` to understand the available columns and how entries are keyed.

In [9]:
df = pd.read_csv(MANIFEST)

print(f"Manifest rows    : {len(df)}")
print(f"Manifest columns : {list(df.columns)}")
print()
print("Column dtypes:")
print(df.dtypes)
print()
print("First 5 rows:")
with pd.option_context('display.max_colwidth', 80):
    print(df[['sr_no','Dialogue_ID','Utterance_ID','split',
              'Emotion','Sentiment','Speaker','path']].head(10).to_string(index=False))
print()
# Split distribution
print("Split distribution:")
print(df['split'].value_counts().to_string())
print()
# Emotion distribution in manifest
print("\nEmotion distribution in manifest:")
print(df['Emotion'].value_counts().to_string())

Manifest rows    : 13707
Manifest columns : ['sr_no', 'Utterance', 'Speaker', 'Emotion', 'Sentiment', 'Dialogue_ID', 'Utterance_ID', 'Season', 'Episode', 'StartTime', 'EndTime', 'split', 'path', 'vlm_analysis']

Column dtypes:
sr_no            int64
Utterance       object
Speaker         object
Emotion         object
Sentiment       object
Dialogue_ID      int64
Utterance_ID     int64
Season           int64
Episode          int64
StartTime       object
EndTime         object
split           object
path            object
vlm_analysis    object
dtype: object

First 5 rows:
 sr_no  Dialogue_ID  Utterance_ID split  Emotion Sentiment         Speaker                                                                  path
     1            0             0 train  neutral   neutral        Chandler /mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/train_splits/dia0_utt0.mp4
     2            0             1 train  neutral   neutral The Interviewer /mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/train_splits/d

## Section 7 — Identify Mapping Keys Between PKL and CSV
The file paths in the manifest follow the pattern `dia{Dialogue_ID}_utt{Utterance_ID}.mp4`.
We reconstruct a canonical key `dia{D}_utt{U}` from both sides and verify alignment.

In [10]:
# ── 7a. Inspect PKL utterance ID format ──────────────────────────────────────
print("=== PKL utterance ID format ===")
# Show the raw keys stored in videoIDs
for vid in all_vids[:5]:
    utts = videoIDs[vid]
    print(f"  dialogue key={repr(vid)}   utt_ids={utts[:5]}  (type={type(utts[0])})")

print()

# ── 7b. Inspect manifest path/key columns ────────────────────────────────────
print("=== Manifest key columns ===")
sample_rows = df[['Dialogue_ID','Utterance_ID','split','path']].head(10)
print(sample_rows.to_string(index=False))
print()
print(f"Dialogue_ID  dtype  : {df['Dialogue_ID'].dtype}")
print(f"Utterance_ID dtype  : {df['Utterance_ID'].dtype}")
print()

# ── 7c. Derive canonical key from manifest ────────────────────────────────────
# The filename stem (dia{D}_utt{U}) uniquely identifies each utterance
df['manifest_key'] = df.apply(
    lambda r: f"dia{int(r['Dialogue_ID'])}_utt{int(r['Utterance_ID'])}", axis=1
)
print("Sample manifest keys:")
print(df[['manifest_key','split','Emotion']].head(10).to_string(index=False))
print()
print(f"Total unique manifest keys : {df['manifest_key'].nunique()}")
print(f"Are all keys unique?       : {df['manifest_key'].nunique() == len(df)}")

# ── 7d. Check PKL key format and reconstruct compound key ─────────────────────
print()
print("=== PKL compound key reconstruction ===")
# videoIDs[dialogue_key] is a list of utterance IDs (may be ints or strings)
# dialogue_key itself may be an int or a string like "0", "train_0", etc.

# Show how PKL keys look for dialogue 0 across train/test
sample_d = all_vids[0]
print(f"PKL dialogue key  : {repr(sample_d)}  (type={type(sample_d).__name__})")
print(f"PKL utt IDs       : {videoIDs[sample_d][:8]}  (type={type(videoIDs[sample_d][0]).__name__})")
print()

# Try to build a PKL compound key assuming dialogue_key is the dialogue_id
# and the position index is the utterance_id
test_key = f"dia{sample_d}_utt{videoIDs[sample_d][0]}"
print(f"Candidate PKL→key (dia{{dia}}_utt{{utt_int}}) : {test_key}")
print(f"  Found in manifest? : {test_key in df['manifest_key'].values}")

=== PKL utterance ID format ===
  dialogue key=0   utt_ids=[0, 1, 2, 3, 4]  (type=<class 'int'>)
  dialogue key=1   utt_ids=[0, 1, 2, 3, 4]  (type=<class 'int'>)
  dialogue key=2   utt_ids=[0, 1, 2, 3, 4]  (type=<class 'int'>)
  dialogue key=3   utt_ids=[0, 1, 2, 3, 4]  (type=<class 'int'>)
  dialogue key=4   utt_ids=[0, 1, 2, 3, 4]  (type=<class 'int'>)

=== Manifest key columns ===
 Dialogue_ID  Utterance_ID split                                                                  path
           0             0 train /mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/train_splits/dia0_utt0.mp4
           0             1 train /mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/train_splits/dia0_utt1.mp4
           0             2 train /mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/train_splits/dia0_utt2.mp4
           0             3 train /mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/train_splits/dia0_utt3.mp4
           0             4 train /mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/train_splits/dia0_utt4

In [11]:
# ── 7e. Build the definitive PKL key-set using all strategies ─────────────────
# Strategy A: key = f"dia{dialogue_key}_utt{utt_id}"   (utt_id from videoIDs list)
# Strategy B: key = f"dia{dialogue_key}_utt{utt_index}"  (position index 0,1,2…)
# Run both and see which achieves higher match coverage

manifest_keys_set = set(df['manifest_key'].values)

# Strategy A
pkl_keys_A = []
for dia_key in all_vids:
    for utt_id in videoIDs[dia_key]:
        pkl_keys_A.append(f"dia{dia_key}_utt{utt_id}")

match_A = sum(1 for k in pkl_keys_A if k in manifest_keys_set)
print(f"Strategy A  (dia_{{dia_key}}_utt_{{utt_id_from_videoIDs}}):")
print(f"  Total PKL utterances  : {len(pkl_keys_A)}")
print(f"  Matched in manifest   : {match_A}  ({100*match_A/len(pkl_keys_A):.1f}%)")
print(f"  Sample keys           : {pkl_keys_A[:5]}")
print()

# Strategy B
pkl_keys_B = []
for dia_key in all_vids:
    for idx in range(len(videoIDs[dia_key])):
        pkl_keys_B.append(f"dia{dia_key}_utt{idx}")

match_B = sum(1 for k in pkl_keys_B if k in manifest_keys_set)
print(f"Strategy B  (dia_{{dia_key}}_utt_{{positional_index}}):")
print(f"  Total PKL utterances  : {len(pkl_keys_B)}")
print(f"  Matched in manifest   : {match_B}  ({100*match_B/len(pkl_keys_B):.1f}%)")
print(f"  Sample keys           : {pkl_keys_B[:5]}")

Strategy A  (dia_{dia_key}_utt_{utt_id_from_videoIDs}):
  Total PKL utterances  : 13708
  Matched in manifest   : 9989  (72.9%)
  Sample keys           : ['dia0_utt0', 'dia0_utt1', 'dia0_utt2', 'dia0_utt3', 'dia0_utt4']

Strategy B  (dia_{dia_key}_utt_{positional_index}):
  Total PKL utterances  : 13708
  Matched in manifest   : 9929  (72.4%)
  Sample keys           : ['dia0_utt0', 'dia0_utt1', 'dia0_utt2', 'dia0_utt3', 'dia0_utt4']


## Section 8 — Validate Mapping Coverage and Mismatches
Using the best mapping strategy from Section 7, determine full/partial coverage and flag any entries that would need a zero-vector fallback.

In [12]:
# Build a VLM lookup using the best strategy, and do per-split coverage analysis
# We try both strategies and report which gives 100% — use that for replacement.

vlm_lookup = dict(zip(df['manifest_key'], df['vlm_analysis']))

print("=" * 65)
print("COVERAGE ANALYSIS — Strategy A  (dia_key + utt_id from videoIDs)")
print("=" * 65)
missing_A, found_A = [], []
records_A = []  # (vid, utt_idx, utt_id, compound_key)

for dia_key in all_vids:
    for utt_idx, utt_id in enumerate(videoIDs[dia_key]):
        key = f"dia{dia_key}_utt{utt_id}"
        records_A.append((dia_key, utt_idx, utt_id, key))
        if key in vlm_lookup:
            found_A.append(key)
        else:
            missing_A.append((dia_key, utt_idx, utt_id, key))

print(f"Total PKL utterances  : {len(records_A)}")
print(f"Found in manifest     : {len(found_A)}  ({100*len(found_A)/len(records_A):.2f}%)")
print(f"Missing from manifest : {len(missing_A)}")
if missing_A:
    print("\nMissing entries (first 20):")
    for dia_key, utt_idx, utt_id, key in missing_A[:20]:
        print(f"  dia={dia_key}  utt_idx={utt_idx}  utt_id={utt_id}  key='{key}'")

print()
print("=" * 65)
print("COVERAGE ANALYSIS — Strategy B  (dia_key + positional index)")
print("=" * 65)
missing_B, found_B = [], []
records_B = []

for dia_key in all_vids:
    for utt_idx in range(len(videoIDs[dia_key])):
        key = f"dia{dia_key}_utt{utt_idx}"
        records_B.append((dia_key, utt_idx, key))
        if key in vlm_lookup:
            found_B.append(key)
        else:
            missing_B.append((dia_key, utt_idx, key))

print(f"Total PKL utterances  : {len(records_B)}")
print(f"Found in manifest     : {len(found_B)}  ({100*len(found_B)/len(records_B):.2f}%)")
print(f"Missing from manifest : {len(missing_B)}")
if missing_B:
    print("\nMissing entries (first 20):")
    for dia_key, utt_idx, key in missing_B[:20]:
        print(f"  dia={dia_key}  utt_idx={utt_idx}  key='{key}'")

COVERAGE ANALYSIS — Strategy A  (dia_key + utt_id from videoIDs)
Total PKL utterances  : 13708
Found in manifest     : 9989  (72.87%)
Missing from manifest : 3719

Missing entries (first 20):
  dia=1039  utt_idx=0  utt_id=0  key='dia1039_utt0'
  dia=1039  utt_idx=1  utt_id=1  key='dia1039_utt1'
  dia=1040  utt_idx=0  utt_id=0  key='dia1040_utt0'
  dia=1040  utt_idx=1  utt_id=1  key='dia1040_utt1'
  dia=1040  utt_idx=2  utt_id=2  key='dia1040_utt2'
  dia=1040  utt_idx=3  utt_id=3  key='dia1040_utt3'
  dia=1040  utt_idx=4  utt_id=4  key='dia1040_utt4'
  dia=1040  utt_idx=5  utt_id=5  key='dia1040_utt5'
  dia=1040  utt_idx=6  utt_id=6  key='dia1040_utt6'
  dia=1040  utt_idx=7  utt_id=7  key='dia1040_utt7'
  dia=1040  utt_idx=8  utt_id=8  key='dia1040_utt8'
  dia=1040  utt_idx=9  utt_id=9  key='dia1040_utt9'
  dia=1040  utt_idx=10  utt_id=10  key='dia1040_utt10'
  dia=1040  utt_idx=11  utt_id=11  key='dia1040_utt11'
  dia=1040  utt_idx=12  utt_id=12  key='dia1040_utt12'
  dia=1040  utt_idx

In [18]:
# ── Split-aware mapping: MELD Dialogue_IDs restart at 0 per split ────────────
# The PKL uses global sequential keys 0..1431, while the manifest uses
# per-split Dialogue_IDs that reset to 0 in each split.
#
# PKL encoding discovered:
#   global_key   0..1038  → MELD train  Dialogue_ID = global_key
#   global_key 1039..1152 → MELD dev    Dialogue_ID = global_key - 1039
#   global_key 1153..1432 → MELD test   Dialogue_ID = global_key - 1153
#
# trainVid contains BOTH train (0-1038) AND dev (1039-1152) keys.

train_keys = sorted(trainVid)
test_keys  = sorted(testVid)

N_TRAIN      = 1039          # MELD standard train dialogue count
N_DEV_OFFSET = 1039          # dev Dialogue_ID offset
N_TEST_OFFSET = min(test_keys)  # = 1153

print(f"trainVid: {len(trainVid)} dialogues, range [{min(train_keys)}, {max(train_keys)}]")
print(f"testVid : {len(testVid)} dialogues, range [{min(test_keys)}, {max(test_keys)}]")
print(f"N_TRAIN={N_TRAIN}, N_DEV_OFFSET={N_DEV_OFFSET}, N_TEST_OFFSET={N_TEST_OFFSET}")
print()

# Build per-split manifest lookups
df_train = df[df['split'] == 'train'].copy()
df_dev   = df[df['split'] == 'dev'].copy()  if 'dev'  in df['split'].values else pd.DataFrame()
df_test  = df[df['split'] == 'test'].copy()

all_splits = df['split'].unique()
print(f"Splits in manifest : {all_splits}")
print(f"Train rows : {len(df_train)}")
print(f"Dev rows   : {len(df_dev)}")
print(f"Test rows  : {len(df_test)}")
print()

def build_vlm_lookup(df_split):
    return {
        f"dia{int(r['Dialogue_ID'])}_utt{int(r['Utterance_ID'])}": r['vlm_analysis']
        for _, r in df_split.iterrows()
    }

vlm_train = build_vlm_lookup(df_train)
vlm_dev   = build_vlm_lookup(df_dev)
vlm_test  = build_vlm_lookup(df_test)

print(f"vlm_train lookup size : {len(vlm_train)}")
print(f"vlm_dev   lookup size : {len(vlm_dev)}")
print(f"vlm_test  lookup size : {len(vlm_test)}")
print()

# Three-way split-aware coverage check
found_sa, missing_sa = 0, []

for dia_key in all_vids:
    for utt_idx, utt_id in enumerate(videoIDs[dia_key]):
        if dia_key < N_TRAIN:               # MELD train
            local_key = f"dia{dia_key}_utt{utt_id}"
            found = local_key in vlm_train
        elif dia_key < N_TEST_OFFSET:       # MELD dev (merged into trainVid)
            local_key = f"dia{dia_key - N_DEV_OFFSET}_utt{utt_id}"
            found = local_key in vlm_dev
        else:                               # MELD test
            local_key = f"dia{dia_key - N_TEST_OFFSET}_utt{utt_id}"
            found = local_key in vlm_test

        if found:
            found_sa += 1
        else:
            missing_sa.append((dia_key, utt_idx, utt_id, local_key))

total = sum(len(videoIDs[v]) for v in all_vids)
print(f"Three-way split-aware coverage:")
print(f"  Total PKL utterances  : {total}")
print(f"  Found                 : {found_sa}  ({100*found_sa/total:.4f}%)")
print(f"  Missing               : {len(missing_sa)}")
if missing_sa:
    print("\n  Missing entries:")
    for dia_key, utt_idx, utt_id, key in missing_sa:
        print(f"    dia_key={dia_key}  utt_idx={utt_idx}  utt_id={utt_id}  key={key!r}")


trainVid: 1152 dialogues, range [0, 1152]
testVid : 280 dialogues, range [1153, 1432]
N_TRAIN=1039, N_DEV_OFFSET=1039, N_TEST_OFFSET=1153

Splits in manifest : ['train' 'dev' 'test']
Train rows : 9989
Dev rows   : 1108
Test rows  : 2610

vlm_train lookup size : 9989
vlm_dev   lookup size : 1108
vlm_test  lookup size : 2610

Three-way split-aware coverage:
  Total PKL utterances  : 13708
  Found                 : 13707  (99.9927%)
  Missing               : 1

  Missing entries:
    dia_key=1149  utt_idx=7  utt_id=7  key='dia110_utt7'


In [19]:
# ── Confirm mapping constants established in the previous cell ────────────────
# These are already set above; just print a summary for reference.
print("Mapping constants confirmed:")
print(f"  N_TRAIN      = {N_TRAIN}   (MELD train dialogs: global keys 0..{N_TRAIN-1})")
print(f"  N_DEV_OFFSET = {N_DEV_OFFSET}   (MELD dev dialogs: global keys {N_DEV_OFFSET}..{N_TEST_OFFSET-1})")
print(f"  N_TEST_OFFSET= {N_TEST_OFFSET}   (MELD test dialogs: global keys {N_TEST_OFFSET}..{max(all_vids)})")
print()
print("VLM lookup sizes:")
print(f"  vlm_train : {len(vlm_train)}")
print(f"  vlm_dev   : {len(vlm_dev)}")
print(f"  vlm_test  : {len(vlm_test)}")
print(f"  total     : {len(vlm_train)+len(vlm_dev)+len(vlm_test)}")
print()
print("Coverage (already computed above):")
print(f"  {found_sa}/{total}  ({100*found_sa/total:.4f}%)")
print(f"  Missing: {len(missing_sa)}")
if missing_sa:
    for dia_key, utt_idx, utt_id, key in missing_sa:
        print(f"    dia_key={dia_key}  utt_id={utt_id}  key={key!r}  → will get zero-vector fallback")


Mapping constants confirmed:
  N_TRAIN      = 1039   (MELD train dialogs: global keys 0..1038)
  N_DEV_OFFSET = 1039   (MELD dev dialogs: global keys 1039..1152)
  N_TEST_OFFSET= 1153   (MELD test dialogs: global keys 1153..1432)

VLM lookup sizes:
  vlm_train : 9989
  vlm_dev   : 1108
  vlm_test  : 2610
  total     : 13707

Coverage (already computed above):
  13707/13708  (99.9927%)
  Missing: 1
    dia_key=1149  utt_id=7  key='dia110_utt7'  → will get zero-vector fallback


In [16]:
# ── Definitive per-split coverage breakdown ───────────────────────────────────
print("=" * 65)
print("DEFINITIVE PER-SPLIT COVERAGE (three-way: train / dev / test)")
print("=" * 65)

for segment_name, seg_keys, lookup, offset in [
    ("TRAIN (global 0-1038)",    [k for k in all_vids if k < N_TRAIN],             vlm_train, 0),
    ("DEV   (global 1039-1152)", [k for k in all_vids if N_TRAIN <= k < N_TEST_OFFSET], vlm_dev,   N_DEV_OFFSET),
    ("TEST  (global 1153-1432)", [k for k in all_vids if k >= N_TEST_OFFSET],       vlm_test,  N_TEST_OFFSET),
]:
    found = missed = 0
    for dia_key in seg_keys:
        for utt_id in videoIDs[dia_key]:
            key = f"dia{dia_key - offset}_utt{utt_id}"
            if key in lookup:
                found += 1
            else:
                missed += 1
    total = found + missed
    pct = 100*found/total if total else 0
    print(f"  {segment_name}: {len(seg_keys)} dialogs, {total} utterances → {found} found ({pct:.2f}%)")

print()
print("=" * 65)
print("MAPPING FORMULA  (global_key → manifest lookup)")
print("=" * 65)
print("  if global_key <  1039:  split='train',  Dialogue_ID = global_key")
print("  if global_key <  1153:  split='dev',    Dialogue_ID = global_key - 1039")
print("  if global_key >= 1153:  split='test',   Dialogue_ID = global_key - 1153")
print()
print("  compound key = f\"dia{Dialogue_ID}_utt{utt_id}\"")
print()
print(f"  One utterance has no VLM text: dia_key=1149 (dev), local='dia110_utt7'")
print(f"  → Will receive zero-vector (768-dim) fallback during replacement.")

print()
print("=" * 65)
print("FINAL PKL INVENTORY")
print("=" * 65)
print(f"  Dialogues        : {len(all_vids)}")
print(f"  Utterances       : {sum(len(videoIDs[v]) for v in all_vids)}")
print(f"  Text dim         : 1024-dim  (float32, 4 BERT variants)")
print(f"  Visual dim       : 342-dim   (float64 ← will become 768-dim Longformer)")
print(f"  Audio dim        : 300-dim   (float32)")
print(f"  Speakers         : 9-dim one-hot (Chandler, Monica, Ross, Rachel, Joey, Phoebe, ...)")
print(f"  Emotion classes  : 7  (0=neutral,1=surprise,2=fear,3=sadness,4=joy,5=disgust,6=anger)")
print(f"  Sentiment classes: 3  (0=negative,1=neutral,2=positive) — stored in videoSentiments")
print(f"  trainVid         : 1152 dialogs (MELD train 0-1038 + dev 0-113)  → 11098 utterances")
print(f"  testVid          :  280 dialogs (MELD test 0-279)                → 2610 utterances")

DEFINITIVE PER-SPLIT COVERAGE (three-way: train / dev / test)
  TRAIN (global 0-1038): 1038 dialogs, 9989 utterances → 9989 found (100.00%)
  DEV   (global 1039-1152): 114 dialogs, 1109 utterances → 1108 found (99.91%)
  TEST  (global 1153-1432): 280 dialogs, 2610 utterances → 2610 found (100.00%)

MAPPING FORMULA  (global_key → manifest lookup)
  if global_key <  1039:  split='train',  Dialogue_ID = global_key
  if global_key <  1153:  split='dev',    Dialogue_ID = global_key - 1039
  if global_key >= 1153:  split='test',   Dialogue_ID = global_key - 1153

  compound key = f"dia{Dialogue_ID}_utt{utt_id}"

  One utterance has no VLM text: dia_key=1149 (dev), local='dia110_utt7'
  → Will receive zero-vector (768-dim) fallback during replacement.

FINAL PKL INVENTORY
  Dialogues        : 1432
  Utterances       : 13708
  Text dim         : 1024-dim  (float32, 4 BERT variants)
  Visual dim       : 342-dim   (float64 ← will become 768-dim Longformer)
  Audio dim        : 300-dim   (float32

In [17]:
# ── Per-split breakdown using the confirmed three-way split-aware mapping ──────
print("=" * 65)
print("PER-SPLIT COVERAGE (three-way split-aware mapping)")
print("=" * 65)

for split_name, split_vids, lookup, offset in [
    ("TRAIN (MELD train, global 0-1038)",       [k for k in trainVid if k < N_TRAIN],            vlm_train, 0),
    ("DEV   (MELD dev merged, global 1039-1152)",[k for k in trainVid if N_TRAIN <= k < N_TEST_OFFSET], vlm_dev,   N_DEV_OFFSET),
    ("TEST  (MELD test, global 1153-1432)",       list(testVid),                                   vlm_test,  N_TEST_OFFSET),
]:
    found = missed = 0
    for dia_key in split_vids:
        for utt_id in videoIDs[dia_key]:
            key = f"dia{dia_key - offset}_utt{utt_id}"
            if key in lookup:
                found += 1
            else:
                missed += 1
    total = found + missed
    pct = 100 * found / total if total else 0
    print(f"  {split_name}")
    print(f"    {found}/{total} utterances found  ({pct:.2f}%)")
    print()

print()
# ── Check for VLM text quality ────────────────────────────────────────────────
print("=" * 65)
print("VLM ANALYSIS TEXT SAMPLES (first 3 entries from manifest)")
print("=" * 65)
for _, row in df.head(3).iterrows():
    print(f"\n  key  : {row['manifest_key']}")
    print(f"  split: {row['split']}  emotion={row['Emotion']}")
    vlm_text = str(row['vlm_analysis'])
    print(f"  vlm  : {vlm_text[:300]}..." if len(vlm_text) > 300 else f"  vlm  : {vlm_text}")

print()

# ── Null / empty VLM check ────────────────────────────────────────────────────
null_count  = df['vlm_analysis'].isna().sum()
empty_count = (df['vlm_analysis'].astype(str).str.strip() == '').sum()
print("=" * 65)
print("VLM TEXT QUALITY CHECK")
print("=" * 65)
print(f"  Null  vlm_analysis entries : {null_count}")
print(f"  Empty vlm_analysis entries : {empty_count}")
print(f"  Valid VLM entries          : {len(df) - null_count}")


PER-SPLIT COVERAGE (three-way split-aware mapping)
  TRAIN (MELD train, global 0-1038)
    9989/9989 utterances found  (100.00%)

  DEV   (MELD dev merged, global 1039-1152)
    1108/1109 utterances found  (99.91%)

  TEST  (MELD test, global 1153-1432)
    2610/2610 utterances found  (100.00%)


VLM ANALYSIS TEXT SAMPLES (first 3 entries from manifest)

  key  : dia0_utt0
  split: train  emotion=neutral
  vlm  : The video begins with both individuals maintaining a neutral expression, characterized by relaxed facial muscles and a steady gaze directed towards each other. As the interaction progresses, Person 1 exhibits subtle head nods and tilts, indicating engagement and attentiveness. His mouth corners rema...

  key  : dia0_utt1
  split: train  emotion=neutral
  vlm  : The man begins with a neutral expression, his mouth corners slightly upturned, suggesting a subtle smile. His head is straight, facing forward, and he maintains direct eye contact with someone off-screen. He is sitting

## Summary — Complete Data Flow: PKL → Model

```
meld_multi_features.pkl  (14 fields)
├── videoIDs        dict[dia_id → list[utt_id]]            # utterance IDs, type TBD after inspection
├── videoSpeakers   dict[dia_id → list[speaker_name]]      # speaker identity
├── videoLabels     dict[dia_id → list[int 0-6]]           # emotion class (ground truth, 7-class)
├── videoSentiments dict[dia_id → list[int 0-2]]           # sentiment (stored directly, not derived)
├── videoText0-3    dict[dia_id → list[float32 (?-dim)]]   # BERT text features (4 variants)
├── videoAudio      dict[dia_id → list[float32 (?-dim)]]   # acoustic features
├── videoVisual     dict[dia_id → list[float32 (?-dim)]]   # ← will be replaced with 768-dim Longformer
├── videoSentence   dict[dia_id → list[str]]               # raw transcriptions
├── trainVid        list[dia_id]                           # train dialogue IDs
├── testVid         list[dia_id]                           # test dialogue IDs
└── _extra          (inspect type after running)

meld_vlm_manifest.csv  mapping strategy:
  PKL compound key = f"dia{dia_key}_utt{utt_id}"   (Strategy A — via videoIDs values)
  OR               = f"dia{dia_key}_utt{utt_index}" (Strategy B — positional index)
  → matched against manifest_key column derived from Dialogue_ID + Utterance_ID

MELDDataset_BERT.__getitem__(index)  →  returns per dialogue:
  [0]  text0        FloatTensor [seq_len, text_dim]   BERT variant 0
  [1]  text1        FloatTensor [seq_len, text_dim]   BERT variant 1
  [2]  text2        FloatTensor [seq_len, text_dim]   BERT variant 2
  [3]  text3        FloatTensor [seq_len, text_dim]   BERT variant 3
  [4]  visual       FloatTensor [seq_len, visual_dim] ← will become 768-dim
  [5]  audio        FloatTensor [seq_len, audio_dim]  acoustic features
  [6]  speakers     FloatTensor [seq_len, spk_dim]    speaker identity
  [7]  mask         FloatTensor [seq_len]              all-ones
  [8]  labels_emo   LongTensor  [seq_len]              emotion class 0-6
  [9]  labels_sen   LongTensor  [seq_len]              sentiment 0=neg,1=neu,2=pos
  [10] vid          str                                dialogue ID
```